# Reproduce the Nature Communications numerical figures

This notebook reproduces the three plots referenced by `NatComm_final.tex` as PNG previews and vector PDF files, and records the parameter choices needed to regenerate the raw data.

There are two paths:

1. **Seeded-data path**: recreates the figures immediately from the tracked tables produced by the canonical seeded run.
2. **Raw-regeneration path**: reruns the Monte Carlo simulations with explicit seeds, sample counts, locality parameters, and Hamiltonian conditioning parameter `c`. These cells are opt-in because the full mode sweep is expensive.

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "qgl_matplotlib_cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display

# Make the helper import work from either the repository root or the manuscript directory.
cwd = Path.cwd().resolve()
for candidate in [cwd, cwd.parent, cwd / "NatCommFinal", cwd.parent / "NatCommFinal"]:
    if (candidate / "qgl_reproduce.py").exists():
        sys.path.insert(0, str(candidate))
        break

import qgl_reproduce as qgl

repo_root = qgl.find_repo_root(cwd)
data_dir = repo_root / "reproduction_data"
seeded_data_dir = repo_root / "reproduction_data_seeded"
plot_dir = repo_root / "reproduced_plots"
data_dir.mkdir(exist_ok=True)
plot_dir.mkdir(exist_ok=True)

# Toggle these to True to regenerate raw simulation data instead of using the tracked seeded tables.
RUN_EXPENSIVE_MODE_SWEEPS = False
RUN_EXPENSIVE_L_SWEEP = False

print(f"Repository root: {repo_root}")
print(f"Data directory:   {data_dir}")
print(f"Plot directory:   {plot_dir}")

## Parameter Choices

The scripts implement the 1D Hamiltonian through the normal chain precision matrix: a tridiagonal matrix with diagonal entries `2` except the last boundary diagonal equal to `1`, and off-diagonal entries `-1`. With inverse temperature `beta=0.5`, this is equivalent to the manuscript convention `H = beta * (NormalPrecision + c I)`.

For the mode-scaling figure the manuscript caption states `N=10^4`, locality `l=3`, and five sample realizations. The old `globalvslocal.jl` currently has `num_averaging_runs=3`, so this notebook pins the caption value `repeats=5` for new raw runs. All derived seeds below use the command-line pipeline's base seed `20240527`.

In [ ]:
BASE_SEED = 20240527

mode_configs = {
    "ill": qgl.ModeSweepConfig(
        condition="ill",
        c=0.0,
        samples=10_000,
        locality=3,
        repeats=5,
        beta=0.5,
        seed=BASE_SEED + 101,
        m_values=tuple(range(100, 1201, 50)),
    ),
    "well": qgl.ModeSweepConfig(
        condition="well",
        c=0.1,
        samples=10_000,
        locality=3,
        repeats=5,
        beta=0.5,
        seed=BASE_SEED + 202,
        m_values=tuple(range(100, 1201, 50)),
    ),
}

l_config = qgl.LSweepConfig(
    c=0.1,
    m=100,
    sample_counts=(10_000, 100_000),
    l_values=(2, 4, 6, 8, 10),
    repeats=1,
    beta=0.5,
    seed=BASE_SEED + 303,
    reuse_samples_across_l=True,
)

pd.DataFrame([
    {"figure": "mode_sweep_ill", **mode_configs["ill"].__dict__},
    {"figure": "mode_sweep_well", **mode_configs["well"].__dict__},
    {"figure": "l_sweep", **l_config.__dict__},
])

## Numerical Sanity Checks

These checks verify the core exact formulas before any sampling is used: the measurement covariance is positive, satisfies the Gaussian uncertainty condition, and exact global inversion recovers the Hamiltonian to machine precision.

In [ ]:
rows = []
for m in [10, 100]:
    sigma = qgl.measurement_covariance(m, c=0.1, beta=0.5)
    quantum_cov = sigma - np.eye(2 * m) / 2
    omega = qgl.omega_matrix(m)
    rows.append({
        "m": m,
        "min_eig_measurement_cov": np.linalg.eigvalsh(sigma).min(),
        "min_eig_quantum_cov_plus_iOmega_over_2": np.linalg.eigvalsh(quantum_cov + 1j * omega / 2).min().real,
        "exact_global_error": qgl.global_reconstruction_error(sigma, m, c=0.1, beta=0.5),
        "classical_exact_error": qgl.classical_exact_error(m, c=0.1, beta=0.5),
    })
pd.DataFrame(rows)

## Tracked seeded tables

The canonical seeded command-line run stores the numerical inputs for all three plots under `reproduction_data_seeded/`.

In [ ]:
seeded_files = sorted(seeded_data_dir.glob("*.csv"))
for path in seeded_files:
    print(path.relative_to(repo_root))

## Mode-Sweep Data

Set `RUN_EXPENSIVE_MODE_SWEEPS = True` in the first cell to regenerate both the ill- and well-conditioned CSVs. The generated files are written incrementally, so an interrupted run still leaves partial data in root-level `reproduction_data/`.

In [ ]:
ill_generated_csv = data_dir / "mode_sweep_ill_generated.csv"
well_generated_csv = data_dir / "mode_sweep_well_generated.csv"

if RUN_EXPENSIVE_MODE_SWEEPS:
    ill_df = qgl.run_mode_sweep(mode_configs["ill"], ill_generated_csv)
    well_df = qgl.run_mode_sweep(mode_configs["well"], well_generated_csv)
else:
    ill_seeded_csv = seeded_data_dir / "mode_sweep_ill_seeded.csv"
    well_seeded_csv = seeded_data_dir / "mode_sweep_well_seeded.csv"
    if not ill_seeded_csv.exists() or not well_seeded_csv.exists():
        raise FileNotFoundError("Run run_seeded_reproduction.py to create the seeded mode-sweep tables.")
    ill_df = pd.read_csv(ill_seeded_csv)
    well_df = pd.read_csv(well_seeded_csv)

display(Markdown(f"Ill-conditioned rows available: **{len(ill_df)}**"))
display(Markdown(f"Well-conditioned rows available: **{len(well_df)}**"))

In [ ]:
ill_plot_stem = plot_dir / "simulation_errors_plot_ill"
ill_plot = ill_plot_stem.with_suffix(".png")
qgl.plot_mode_sweep(ill_df, "ill-conditioned", ill_plot_stem)
plt.close("all")
display(Image(filename=str(ill_plot)))

well_plot_stem = plot_dir / "simulation_errors_plot_well"
well_plot = well_plot_stem.with_suffix(".png")
qgl.plot_mode_sweep(well_df, "well-conditioned", well_plot_stem)
plt.close("all")
display(Image(filename=str(well_plot)))

## Locality-Sweep Data

Set `RUN_EXPENSIVE_L_SWEEP = True` in the first cell to regenerate the sampled `N=10^4` and `N=10^5` curves and the exact-covariance inset. Without that flag, the notebook loads the tracked tables from the canonical seeded run and writes matching PNG and vector PDF figures immediately.

The exact-covariance curve and both reference baselines are loaded from the same seeded run as the sampled curves.

In [ ]:
l_sampled_csv = data_dir / "l_sweep_sampled_generated.csv"
l_exact_csv = data_dir / "l_sweep_exact_generated.csv"

if RUN_EXPENSIVE_L_SWEEP:
    sampled_l_df = qgl.run_l_sweep(l_config, l_sampled_csv)
    exact_l_df = qgl.exact_l_sweep(l_config)
    exact_l_df.to_csv(l_exact_csv, index=False)

    rng = np.random.default_rng(BASE_SEED + 404)
    global_errors = []
    for _ in range(l_config.repeats):
        samples = qgl.sample_measurements(l_config.m, 100_000, l_config.c, rng, l_config.beta)
        cov_est = qgl.covariance_estimate(samples)
        global_errors.append(qgl.global_reconstruction_error(cov_est, l_config.m, l_config.c, l_config.beta))
    global_sampled_error = float(np.mean(global_errors))
    classical_exact_error = qgl.classical_exact_error(l_config.m, l_config.c, l_config.beta)
else:
    sampled_l_df = pd.read_csv(seeded_data_dir / "l_sweep_sampled_seeded.csv")
    exact_l_df = pd.read_csv(seeded_data_dir / "l_sweep_exact_seeded.csv")
    baselines_df = pd.read_csv(seeded_data_dir / "l_sweep_baselines_seeded.csv")
    global_sampled_error = float(baselines_df.loc[0, "GlobalSampledError"])
    classical_exact_error = float(baselines_df.loc[0, "ClassicalExactError"])

display(sampled_l_df)
display(exact_l_df)

In [ ]:
l_plot_stem = plot_dir / "improved_plot"
l_plot = l_plot_stem.with_suffix(".png")
qgl.plot_l_sweep(sampled_l_df, exact_l_df, global_sampled_error, classical_exact_error, l_plot_stem)
plt.close("all")
display(Image(filename=str(l_plot)))

## Ill/Well First-Mode Window Heatmaps

This compact check reruns the first five modes for the ill-conditioned chain (`c = 0`) and the well-conditioned chain (`c = 0.1`) with the same seed. The side-by-side heatmaps make the condition-dependent reconstruction differences visible without storing the full mode-sweep matrix dump.


In [ ]:
import compare_condition_window_heatmaps as condition_windows

condition_window_summary = condition_windows.run_condition_window_checks(
    seed=100,
    m=100,
    samples=100_000,
    beta=0.5,
    locality=4,
    window_modes=5,
    start_mode=1,
    output_dir=repo_root / "condition_window_checks",
)

condition_metrics = pd.DataFrame([
    {
        "condition": item["condition"],
        "c": item["c"],
        "local_max_abs_error": item["local_metrics"]["max_abs_error"],
        "global_max_abs_error": item["global_metrics"]["max_abs_error"],
        "target_max_abs": item["local_metrics"]["target_max_abs"],
    }
    for item in condition_window_summary["conditions"]
])
display(condition_metrics)

display(Image(filename=condition_window_summary["error_heatmap"]))
display(Image(filename=condition_window_summary["target_heatmap"]))


## Historical figure provenance

The tracked seeded tables are sufficient to regenerate the current figures as PNG previews and vector PDFs. The root-level mode-sweep figure copies mirror these canonical outputs for compatibility; historical hard-coded values remain in compatibility helpers and legacy plotting scripts, and are not used by the canonical seeded workflow.

For new results, rerun the canonical seeded command-line pipeline and use the resulting tables and figures together.